# 16 — Performance stats & figures

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = next(p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents]
                    if (p / "analysis" / "lib").exists())
sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
import seaborn as sns

from analysis.lib.stats import nadeau_bengio_test, bh_fdr
from analysis.lib.plotting import (GROUP_ORDER, PALETTE, DISPLAY_NAMES,
                                   VARIABLE_LABELS, compute_nb_sig_table,
                                   compute_wilcoxon_sig_table, get_sig_flags,
                                   plot_sig_markers, sig_legend_entries)

CORE_OUT = PROJECT_ROOT / "outputs" / "predictive_modeling_core"
POOLED  = CORE_OUT / "pooled"
LOCO    = CORE_OUT / "loco"
FIG_DIR = CORE_OUT / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

# subject counts come from modeling_summary.csv (no clinical data needed here)

In [ ]:
pooled_summary = pd.read_csv(POOLED / "modeling_summary.csv")
pooled_scores  = pd.read_csv(POOLED / "all_scores.csv")
pooled_stacker = pd.read_csv(POOLED / "stacking_scores.csv")

loco_scores  = pd.read_csv(LOCO / "all_scores.csv")
loco_stacker = pd.read_csv(LOCO / "stacking_scores.csv")

SINGLE_COHORT = "SB"
SINGLE_COHORT_DISPLAY = "Bundang VCI"
SINGLE_DIR = CORE_OUT / f"single_cohort_{SINGLE_COHORT}"
if (SINGLE_DIR / "all_scores.csv").exists():
    single_scores  = pd.read_csv(SINGLE_DIR / "all_scores.csv")
    single_stacker = pd.read_csv(SINGLE_DIR / "stacking_scores.csv")
    print(f"single-cohort ({SINGLE_COHORT}) outputs loaded")
else:
    single_scores = single_stacker = None
    print(f"single-cohort outputs not found at {SINGLE_DIR} — reference bar omitted")

BASE_MODELS = {
    ("Majority", "base"):   "Majority",
    ("Permuted", "base"):   "Permuted",
    ("Baseline", "base"):   "Baseline",
    ("Acute",    "Lesion"): "Lesion",
    ("Acute",    "fLNM"):   "fLNM",
    ("Acute",    "sLNM"):   "sLNM",
}


def to_long(scores_df, stacker_df, metric="AUPRC"):
    rows = []
    for (model, modality), group in BASE_MODELS.items():
        sub = scores_df[(scores_df["Model"] == model) & (scores_df["Modality"] == modality)]
        for _, r in sub.iterrows():
            rows.append({"Target": r["Target"], "Group": group, "Fold": r["Fold"],
                         "AUPRC": r[metric], "AUPRC_train": r.get(f"{metric}_train", np.nan)})
    for _, r in stacker_df.iterrows():
        rows.append({"Target": r["Target"], "Group": "Stacking", "Fold": r["Fold"],
                     "AUPRC": r[metric], "AUPRC_train": r.get(f"{metric}_train", np.nan)})
    return pd.DataFrame(rows)


pooled_long = to_long(pooled_scores, pooled_stacker)
loco_long   = to_long(loco_scores,   loco_stacker)
single_long = (to_long(single_scores, single_stacker)
               if single_scores is not None else None)

nb_pooled = compute_nb_sig_table(pooled_long, n_splits=10, test_size=0.1,
                                 focal_groups=["Stacking"])
# baseline vs null models
nb_pooled_base = compute_nb_sig_table(pooled_long, n_splits=10, test_size=0.1,
                                      focal_groups=["Baseline"],
                                      reference_groups=["Majority", "Permuted"])
nb_pooled = pd.concat([nb_pooled, nb_pooled_base], ignore_index=True)
nb_loco_rows = []
for tgt, sub in loco_long.groupby("Target"):
    if sub["Fold"].nunique() < 2:
        continue
    nb_loco_rows.append(compute_wilcoxon_sig_table(sub, focal_groups=["Stacking"]))
    nb_loco_rows.append(compute_wilcoxon_sig_table(sub, focal_groups=["Baseline"],
                                                   reference_groups=["Majority", "Permuted"]))
nb_loco = pd.concat(nb_loco_rows, ignore_index=True) if nb_loco_rows else pd.DataFrame()
nb_pooled.to_csv(FIG_DIR / "nb_significance_pooled.csv", index=False)
print(f"pooled NB tests: {len(nb_pooled)} | LOCO Wilcoxon tests: {len(nb_loco)}")

_summ = pd.read_csv(POOLED / "modeling_summary.csv")
n_per_target = dict(zip(_summ["Target"], _summ["N"].astype(int)))

In [ ]:
# Pooled CV violin (Fig 2a input)
CORE_GROUPS = ["Majority", "Permuted", "Baseline", "Stacking"]


def plot_pooled_violin(long_df, stacker_df, scores_df, nb_table, ylabel, out_stem):
    sns.set_theme(style="whitegrid", font_scale=1.3,
                  rc={"font.weight": "bold", "axes.labelweight": "bold",
                      "axes.titleweight": "bold"})
    fig, ax = plt.subplots(figsize=(20, 7))
    subset = long_df[long_df["Group"].isin(CORE_GROUPS)]
    sns.violinplot(data=subset, x="Target", y="AUPRC", hue="Group",
                   hue_order=CORE_GROUPS, palette=PALETTE,
                   linewidth=0, saturation=0.7, cut=0, inner=None, width=0.85, ax=ax)

    targets_list = [t for t in VARIABLE_LABELS if t in subset["Target"].unique()]
    n_targets = len(targets_list)
    n_groups = len(CORE_GROUPS)
    group_w = 0.85
    band_w = group_w / n_groups
    offsets = np.linspace(-group_w / 2 + band_w / 2, group_w / 2 - band_w / 2, n_groups)

    train_means = long_df.groupby(["Target", "Group"])["AUPRC_train"].mean()

    ax.set_ylim(0, 1.18)
    sig_base_y = 1.02
    sig_step = 0.045
    rng_jitter = np.random.RandomState(0)

    for x_idx, tname in enumerate(targets_list):
        for g_idx, group in enumerate(CORE_GROUPS):
            cx = x_idx + offsets[g_idx]
            vals = long_df.loc[(long_df["Target"] == tname)
                               & (long_df["Group"] == group), "AUPRC"].values
            if len(vals):
                jx = rng_jitter.normal(loc=cx, scale=0.015, size=len(vals))
                ax.scatter(jx, vals, s=8, color="gray", alpha=0.35, zorder=2)
                ax.scatter(cx, np.nanmean(vals), color="#222", marker="D", s=55, zorder=3)
                tr = train_means.get((tname, group), np.nan)
                if np.isfinite(tr):
                    ax.scatter(cx, tr, s=55, zorder=3, facecolors="none",
                               edgecolors="#222", linewidths=1.4, marker="o")
            if group in ("Baseline", "Stacking") and len(nb_table):
                sig_flags = get_sig_flags(nb_table, tname, group)
                plot_sig_markers(ax, cx, sig_flags, sig_base_y, sig_step,
                                 marker_size=70, color=PALETTE[group])
            if len(vals):
                ax.text(cx, sig_base_y - 0.035, f"{np.nanmean(vals):.2f}",
                        ha="center", va="top", fontsize=13, fontweight="bold",
                        color="#222", zorder=5)

    ax.set_xticks(range(n_targets))
    ax.set_xticklabels([f"{VARIABLE_LABELS[t]}\n(n={n_per_target[t]})"
                        for t in targets_list], fontsize=13, rotation=0)
    ax.set_ylabel(ylabel)
    ax.set_xlabel("")

    extra_handles = [
        Line2D([0], [0], marker="D", color="w", markerfacecolor="#222", markersize=9,
               linewidth=0, label="Mean test"),
        Line2D([0], [0], marker="o", color="w", markerfacecolor="none",
               markeredgecolor="#222", markersize=9, markeredgewidth=1.4,
               linewidth=0, label="Mean train"),
    ] + sig_legend_entries()
    mh, ml = ax.get_legend_handles_labels()
    ax.legend(mh + extra_handles,
              [DISPLAY_NAMES.get(l, l) for l in ml]
              + [h.get_label() for h in extra_handles],
              bbox_to_anchor=(0.5, -0.20), loc="upper center",
              frameon=True, ncol=3, fontsize=12)
    plt.tight_layout()
    plt.subplots_adjust(bottom=0.22)
    for ext in (".png", ".svg", ".pdf"):
        fig.savefig(FIG_DIR / f"{out_stem}{ext}",
                    **(dict(dpi=200) if ext == ".png" else {}), bbox_inches="tight")
    print(f"saved {FIG_DIR / (out_stem + '.png')}")


plot_pooled_violin(pooled_long, pooled_stacker, pooled_scores, nb_pooled,
                   "Predictive performance (AUPRC)", "pooled_cv_violin")

In [ ]:
# LOCO panels (raw + prevalence-normalised, Fig S12a input)
LOCO_GROUPS = ["Majority", "Permuted", "Baseline", "Stacking"]


def plot_loco_panel(loco_vals_df, value_col, pooled_ref, single_ref, nb_table,
                    ylabel, out_stem, ylim, sig_base_y, mean_label_groups=None,
                    ref_label_suffix="mean"):
    sns.set_theme(style="whitegrid", font_scale=1.3,
                  rc={"font.weight": "bold", "axes.labelweight": "bold",
                      "axes.titleweight": "bold"})
    fig, ax = plt.subplots(figsize=(16, 7) if value_col == "AUPRC" else (22, 7))
    targets_list = [t for t in VARIABLE_LABELS if t in loco_vals_df["Target"].unique()]
    n_targets = len(targets_list)
    n_groups = len(LOCO_GROUPS)
    group_w = 0.85
    band_w = group_w / n_groups
    offsets = np.linspace(-group_w / 2 + band_w / 2, group_w / 2 - band_w / 2, n_groups)

    ax.set_ylim(*ylim)
    if ylim[0] < 0:
        ax.axhline(0, color="black", lw=0.8, ls="-", alpha=0.6, zorder=1)
    sig_step = 0.045
    rng_jitter = np.random.RandomState(0)
    n_cohorts_per_target = {
        t: int(loco_vals_df.loc[loco_vals_df["Target"] == t, "Fold"].nunique())
        for t in targets_list}

    for x_idx, tname in enumerate(targets_list):
        for g_idx, group in enumerate(LOCO_GROUPS):
            cx = x_idx + offsets[g_idx]
            color = PALETTE[group]
            prow = pooled_ref[(pooled_ref["Target"] == tname)
                              & (pooled_ref["Group"] == group)]
            if len(prow) and pd.notna(prow.iloc[0, -1]):
                ax.hlines(float(prow.iloc[0, -1]), cx - band_w * 0.45,
                          cx + band_w * 0.45, colors="black", linewidth=3, zorder=5)
            if single_ref is not None:
                srow = single_ref[(single_ref["Target"] == tname)
                                  & (single_ref["Group"] == group)]
                if len(srow) and pd.notna(srow.iloc[0, -1]):
                    ax.hlines(float(srow.iloc[0, -1]), cx - band_w * 0.45,
                              cx + band_w * 0.45, colors="#555555", linewidth=2,
                              linestyle="--", zorder=5)
            sub = loco_vals_df[(loco_vals_df["Target"] == tname)
                               & (loco_vals_df["Group"] == group)]
            vals = sub[value_col].dropna().values
            if len(vals):
                jx = rng_jitter.normal(loc=cx, scale=band_w * 0.12, size=len(vals))
                ax.scatter(jx, vals, s=42, color=color, alpha=0.85,
                           edgecolor="black", linewidth=0.5, zorder=3)
                ax.scatter(cx, np.nanmean(vals), s=110, marker="D", color=color,
                           edgecolor="black", linewidth=1.4, zorder=4)
            if group in ("Baseline", "Stacking") and len(nb_table):
                sig_flags = get_sig_flags(nb_table, tname, group)
                plot_sig_markers(ax, cx, sig_flags, sig_base_y, sig_step,
                                 marker_size=70, color=color)
            show_label = (mean_label_groups is None or group in mean_label_groups)
            if len(vals) and show_label:
                ax.text(cx, sig_base_y - 0.040, f"{np.nanmean(vals):.2f}",
                        ha="center", va="top", fontsize=10, fontweight="bold",
                        color="#222", zorder=6)

    for x in range(1, n_targets):
        ax.axvline(x - 0.5, color="gray", linewidth=0.3, alpha=0.4)
    ax.set_xticks(range(n_targets))
    ax.set_xticklabels([f"{VARIABLE_LABELS[t]}\n(n={n_cohorts_per_target[t]})"
                        for t in targets_list], fontsize=13, rotation=0)
    ax.set_ylabel(ylabel)
    ax.set_xlabel("")

    model_handles = [Line2D([0], [0], marker="o", color="w",
                            markerfacecolor=PALETTE[g], markeredgecolor="black",
                            markersize=9, label=DISPLAY_NAMES.get(g, g))
                     for g in LOCO_GROUPS]
    extra_handles = [
        Line2D([0], [0], color="black", linewidth=3,
               label=f"Pooled CV {ref_label_suffix}"),
        Line2D([0], [0], color="#555555", linewidth=2, linestyle="--",
               label=f"Single cohort ({SINGLE_COHORT_DISPLAY}) {ref_label_suffix}"),
        Line2D([0], [0], marker="D", color="w", markerfacecolor="#888",
               markeredgecolor="black", markersize=9, linewidth=0,
               label="Mean across cohorts"),
    ] + sig_legend_entries(test_name="Wilcoxon")
    ax.legend(model_handles + extra_handles,
              [h.get_label() for h in model_handles + extra_handles],
              bbox_to_anchor=(0.5, -0.20), loc="upper center",
              frameon=True, ncol=3, fontsize=12)
    plt.tight_layout()
    for ext in (".png", ".svg", ".pdf"):
        fig.savefig(FIG_DIR / f"{out_stem}{ext}",
                    **(dict(dpi=200) if ext == ".png" else {}), bbox_inches="tight")
    print(f"saved {FIG_DIR / (out_stem + '.png')}")


pooled_means = (pooled_long.groupby(["Target", "Group"])["AUPRC"].mean()
                .rename("ref").reset_index())
single_means = (single_long.groupby(["Target", "Group"])["AUPRC"].mean()
                .rename("ref").reset_index() if single_long is not None else None)

plot_loco_panel(loco_long, "AUPRC", pooled_means, single_means, nb_loco,
                "Predictive performance (AUPRC)", "loco_cv_panel",
                ylim=(0, 1.18), sig_base_y=1.02)

In [ ]:
# Prevalence-normalised LOCO panel
def normalise(x, prev):
    if pd.isna(x) or pd.isna(prev) or (1.0 - prev) < 1e-9:
        return np.nan
    return (x - prev) / (1.0 - prev)


target_cols = [t for t in VARIABLE_LABELS if t in loco_long["Target"].unique()]
prev_pooled = (pooled_long[pooled_long["Group"] == "Majority"]
               .groupby("Target")["AUPRC"].mean().to_dict())
prev_single = (single_long[single_long["Group"] == "Majority"]
               .groupby("Target")["AUPRC"].mean().to_dict()
               if single_long is not None else {t: np.nan for t in target_cols})

loco_long_norm = loco_long.copy()
majority_floor = (loco_long[loco_long["Group"] == "Majority"]
                  .set_index(["Target", "Fold"])["AUPRC"].to_dict())
loco_long_norm["prev"] = loco_long_norm.apply(
    lambda r: majority_floor.get((r["Target"], r["Fold"]), np.nan), axis=1)
loco_long_norm["norm_AUPRC"] = loco_long_norm.apply(
    lambda r: normalise(r["AUPRC"], r["prev"]), axis=1)
loco_long_norm.to_csv(LOCO / "loco_per_cohort_normauprc.csv", index=False)

pooled_means_norm = pd.DataFrame([
    {"Target": tgt, "Group": grp,
     "ref": normalise(float(pooled_means[(pooled_means["Target"] == tgt)
                                         & (pooled_means["Group"] == grp)]["ref"].iloc[0]),
                      prev_pooled[tgt])}
    for tgt in target_cols for grp in LOCO_GROUPS
    if len(pooled_means[(pooled_means["Target"] == tgt) & (pooled_means["Group"] == grp)])])

single_means_norm = None
if single_means is not None:
    single_means_norm = pd.DataFrame([
        {"Target": tgt, "Group": grp,
         "ref": normalise(float(single_means[(single_means["Target"] == tgt)
                                             & (single_means["Group"] == grp)]["ref"].iloc[0]),
                          prev_single[tgt])}
        for tgt in target_cols for grp in LOCO_GROUPS
        if len(single_means[(single_means["Target"] == tgt) & (single_means["Group"] == grp)])])

# Wilcoxon markers on the plotted (normalised) scale
nb_loco_norm_rows = []
for tgt, sub in loco_long_norm.groupby("Target"):
    if sub["Fold"].nunique() < 2:
        continue
    nb_loco_norm_rows.append(compute_wilcoxon_sig_table(sub, focal_groups=["Stacking"],
                                                        metric="norm_AUPRC"))
    nb_loco_norm_rows.append(compute_wilcoxon_sig_table(sub, focal_groups=["Baseline"],
                                                        reference_groups=["Majority", "Permuted"],
                                                        metric="norm_AUPRC"))
nb_loco_norm = pd.concat(nb_loco_norm_rows, ignore_index=True)

plot_loco_panel(loco_long_norm, "norm_AUPRC", pooled_means_norm, single_means_norm,
                nb_loco_norm, "Predictive performance\n(normalised AUPRC)",
                "loco_cv_panel_normauprc", ylim=(-0.30, 0.8), sig_base_y=0.67,
                mean_label_groups=("Baseline", "Stacking"),
                ref_label_suffix="(normalised)")

In [ ]:
# Drop-one-view ablation (Fig 2b input)
dropone_scores = pd.read_csv(POOLED / "stacking_dropone_scores.csv")
full_auprc = pooled_stacker.set_index(["Target", "Fold"])["AUPRC"]

rows = []
for _, r in dropone_scores.iterrows():
    full = full_auprc.get((r["Target"], r["Fold"]))
    if pd.notna(full) and pd.notna(r["AUPRC"]):
        rows.append({"Target": r["Target"], "DroppedView": r["DroppedView"],
                     "Fold": r["Fold"], "AUPRC_drop": float(full) - float(r["AUPRC"])})
drop_df = pd.DataFrame(rows)
summary = (drop_df.groupby(["Target", "DroppedView"])["AUPRC_drop"]
           .agg(["mean", "sem"]).reset_index())

DROP_VIEW_ORDER = ["lesion_acute", "flnm_acute", "slnm_acute"]
DROP_VIEW_LABEL = {"lesion_acute": "Lesion location", "flnm_acute": "fLNM",
                   "slnm_acute": "sLNM"}
DROP_VIEW_COLOR = {"lesion_acute": PALETTE["Lesion"], "flnm_acute": PALETTE["fLNM"],
                   "slnm_acute": PALETTE["sLNM"]}

targets_order = [t for t in VARIABLE_LABELS if t in summary["Target"].unique()]
n_targets = len(targets_order)
n_views = len(DROP_VIEW_ORDER)
group_w = 0.85
band_w = group_w / n_views
offsets = np.linspace(-group_w / 2 + band_w / 2, group_w / 2 - band_w / 2, n_views)

sns.set_theme(style="whitegrid", font_scale=1.3,
              rc={"font.weight": "bold", "axes.labelweight": "bold",
                  "axes.titleweight": "bold"})
fig, ax = plt.subplots(figsize=(max(10, 1.8 * n_targets), 6))
for x_idx, tname in enumerate(targets_order):
    for g_idx, view in enumerate(DROP_VIEW_ORDER):
        cx = x_idx + offsets[g_idx]
        row = summary[(summary["Target"] == tname) & (summary["DroppedView"] == view)]
        if row.empty:
            continue
        mean = float(row["mean"].iloc[0])
        sem = float(row["sem"].iloc[0]) if pd.notna(row["sem"].iloc[0]) else 0.0
        ax.bar(cx, mean, width=band_w * 0.95, color=DROP_VIEW_COLOR[view],
               edgecolor="black", linewidth=0.6,
               label=DROP_VIEW_LABEL[view] if x_idx == 0 else None)
        if sem > 0:
            ax.errorbar(cx, mean, yerr=sem, color="black", linewidth=0.8,
                        capsize=3, zorder=4)
ax.axhline(0, color="black", linewidth=0.7)
for x in range(1, n_targets):
    ax.axvline(x - 0.5, color="gray", linewidth=0.3, alpha=0.4)
ax.set_xticks(range(n_targets))
ax.set_xticklabels([VARIABLE_LABELS[t] for t in targets_order], fontsize=17, rotation=0)
ax.set_ylabel("Performance difference \n(ΔAUPRC, full − leave-one-set-out)",
              fontsize=11, fontweight="bold")
ax.set_xlabel("")
ax.legend(bbox_to_anchor=(0.5, -0.10), loc="upper center", ncol=4,
          title="Feature set dropped", frameon=True, fontsize=10)
plt.tight_layout()
for ext in (".png", ".svg", ".pdf"):
    fig.savefig(FIG_DIR / f"stacker_dropone_ablation{ext}",
                **(dict(dpi=200) if ext == ".png" else {}), bbox_inches="tight")
print(f"saved {FIG_DIR / 'stacker_dropone_ablation.png'}")

In [ ]:
# Modality comparison violin (Fig S4)
MOD_GROUPS = ["Lesion", "fLNM", "sLNM"]
nb_mod_vs_nulls = compute_nb_sig_table(
    pooled_long, n_splits=10, test_size=0.1,
    focal_groups=MOD_GROUPS, reference_groups=["Majority", "Permuted", "Baseline"])

PAIRWISE = [("sLNM", "Lesion"), ("fLNM", "Lesion"), ("sLNM", "fLNM")]
rows = []
for tgt in pooled_long["Target"].unique():
    sub = pooled_long[pooled_long["Target"] == tgt]
    target_pvals, target_idx = [], []
    for focal, reference in PAIRWISE:
        f = sub[sub["Group"] == focal].set_index("Fold")["AUPRC"]
        r = sub[sub["Group"] == reference].set_index("Fold")["AUPRC"]
        common = f.index.intersection(r.index)
        if len(common) < 2:
            continue
        out = nadeau_bengio_test(f.loc[common].values, r.loc[common].values,
                                 n_splits=10, test_size=0.1, alternative="greater")
        rows.append({"Target": tgt, "Focal": focal, "Reference": reference,
                     "mean_diff": out["mean_diff"], "t_stat": out["t_stat"],
                     "p": out["p"], "n_paired": out["n_paired"]})
        target_pvals.append(out["p"])
        target_idx.append(len(rows) - 1)
    if target_pvals:
        adj = bh_fdr(np.asarray(target_pvals, dtype=float))
        for i, row_idx in enumerate(target_idx):
            rows[row_idx]["p_adj_BH"] = float(adj[i])
nb_cross = pd.DataFrame(rows)

sns.set_theme(style="whitegrid", font_scale=1.3,
              rc={"font.weight": "bold", "axes.labelweight": "bold",
                  "axes.titleweight": "bold"})
fig, ax = plt.subplots(figsize=(20, 7))
pooled_mod = pooled_long[pooled_long["Group"].isin(MOD_GROUPS)]
sns.violinplot(data=pooled_mod, x="Target", y="AUPRC", hue="Group",
               hue_order=MOD_GROUPS, palette=PALETTE,
               linewidth=0, saturation=0.7, cut=0, inner=None, width=0.7, ax=ax)

targets_list = [t for t in VARIABLE_LABELS if t in pooled_mod["Target"].unique()]
n_targets = len(targets_list)
n_groups = len(MOD_GROUPS)
group_w = 0.7
band_w = group_w / n_groups
offsets = np.linspace(-group_w / 2 + band_w / 2, group_w / 2 - band_w / 2, n_groups)

ax.set_ylim(0, 1.28)
ax.set_yticks([0, 0.2, 0.4, 0.6, 0.8, 1.0])
sig_base_y = 1.02
sig_step = 0.045
CROSS_MARKERS_PER_FOCAL = {
    "fLNM": [("Lesion", (6, 0, 0))],
    "sLNM": [("Lesion", (6, 0, 0)), ("fLNM", (7, 0, 0))],
}
N_NULL_MARKERS = 3
train_means = pooled_long.groupby(["Target", "Group"])["AUPRC_train"].mean()

rng_jitter = np.random.RandomState(0)
for x_idx, tname in enumerate(targets_list):
    for g_idx, group in enumerate(MOD_GROUPS):
        cx = x_idx + offsets[g_idx]
        vals = pooled_long.loc[(pooled_long["Target"] == tname)
                               & (pooled_long["Group"] == group), "AUPRC"].values
        if len(vals):
            jx = rng_jitter.normal(loc=cx, scale=0.012, size=len(vals))
            ax.scatter(jx, vals, s=8, color="gray", alpha=0.35, zorder=2)
            ax.scatter(cx, np.nanmean(vals), color="#222", marker="D", s=55, zorder=3)
            tr = train_means.get((tname, group), np.nan)
            if pd.notna(tr):
                ax.scatter(cx, tr, s=55, zorder=3, facecolors="none",
                           edgecolors="#222", linewidths=1.4, marker="o")
        sig_flags = get_sig_flags(nb_mod_vs_nulls, tname, group)
        plot_sig_markers(ax, cx, sig_flags, sig_base_y, sig_step,
                         marker_size=65, color=PALETTE[group])
        if len(vals):
            ax.text(cx, sig_base_y - 0.035, f"{np.nanmean(vals):.2f}",
                    ha="center", va="top", fontsize=13, fontweight="bold",
                    color="#222", zorder=5)
        for cm_idx, (ref, marker_spec) in enumerate(CROSS_MARKERS_PER_FOCAL.get(group, [])):
            row = nb_cross[(nb_cross["Target"] == tname)
                           & (nb_cross["Focal"] == group)
                           & (nb_cross["Reference"] == ref)]
            if not len(row):
                continue
            p_adj = (row["p_adj_BH"].iloc[0] if "p_adj_BH" in row.columns
                     else row["p"].iloc[0])
            d = row["mean_diff"].iloc[0]
            if pd.notna(p_adj) and p_adj < 0.05 and pd.notna(d) and d > 0:
                y = sig_base_y + (N_NULL_MARKERS + cm_idx) * sig_step
                ax.scatter(cx, y, marker=marker_spec, s=85, color=PALETTE[group],
                           edgecolor="black", linewidth=0.8, zorder=6, clip_on=False)

ax.set_xticks(range(n_targets))
ax.set_xticklabels([VARIABLE_LABELS[t] for t in targets_list], fontsize=13, rotation=0)
ax.set_ylabel("Predictive performance (AUPRC)")
ax.set_xlabel("")
extra = [
    Line2D([0], [0], marker="D", color="w", markerfacecolor="#222", markersize=9,
           linewidth=0, label="Mean test AUPRC"),
    Line2D([0], [0], marker="o", color="w", markerfacecolor="none",
           markeredgecolor="#222", markersize=9, markeredgewidth=1.4, linewidth=0,
           label="Mean train AUPRC"),
] + sig_legend_entries() + [
    Line2D([0], [0], marker=(6, 0, 0), color="w", markerfacecolor="black",
           markeredgecolor="black", markersize=10, linewidth=0,
           label=r"> Lesion location ($p_{FDR}<0.05$, NB)"),
    Line2D([0], [0], marker=(7, 0, 0), color="w", markerfacecolor="black",
           markeredgecolor="black", markersize=10, linewidth=0,
           label=r"> fLNM ($p_{FDR}<0.05$, NB)"),
]
mh, ml = ax.get_legend_handles_labels()
MODALITY_LEGEND = {"Lesion": "Lesion location",
                   "fLNM": "Functional lesion network mapping",
                   "sLNM": "Structural lesion network mapping"}
ml = [MODALITY_LEGEND.get(l, l) for l in ml]
ax.legend(mh + extra, ml + [h.get_label() for h in extra],
          bbox_to_anchor=(0.5, -0.10), loc="upper center", ncol=5,
          frameon=True, fontsize=12)
plt.tight_layout()
for ext in (".png", ".svg", ".pdf"):
    fig.savefig(FIG_DIR / f"modality_comparison_violin{ext}",
                **(dict(dpi=200) if ext == ".png" else {}), bbox_inches="tight")
print(f"saved {FIG_DIR / 'modality_comparison_violin.png'}")

In [ ]:
# AUROC siblings: pooled violin (Fig S2) + LOCO panel (Fig S13)
pooled_long_auroc = to_long(pooled_scores, pooled_stacker, metric="AUROC")
loco_long_auroc   = to_long(loco_scores,   loco_stacker,   metric="AUROC")
single_long_auroc = (to_long(single_scores, single_stacker, metric="AUROC")
                     if single_scores is not None else None)

nb_pooled_auroc = compute_nb_sig_table(pooled_long_auroc, n_splits=10,
                                       test_size=0.1, focal_groups=["Stacking"])
nb_pooled_auroc = pd.concat([nb_pooled_auroc, compute_nb_sig_table(
    pooled_long_auroc, n_splits=10, test_size=0.1, focal_groups=["Baseline"],
    reference_groups=["Majority", "Permuted"])], ignore_index=True)
nb_loco_auroc_rows = []
for tgt, sub in loco_long_auroc.groupby("Target"):
    if sub["Fold"].nunique() < 2:
        continue
    nb_loco_auroc_rows.append(compute_wilcoxon_sig_table(sub, focal_groups=["Stacking"]))
    nb_loco_auroc_rows.append(compute_wilcoxon_sig_table(sub, focal_groups=["Baseline"],
                                                         reference_groups=["Majority", "Permuted"]))
nb_loco_auroc = (pd.concat(nb_loco_auroc_rows, ignore_index=True)
                 if nb_loco_auroc_rows else pd.DataFrame())

plot_pooled_violin(pooled_long_auroc, pooled_stacker, pooled_scores,
                   nb_pooled_auroc, "Predictive performance (AUROC)",
                   "pooled_cv_violin_auroc")

pooled_means_auroc = (pooled_long_auroc.groupby(["Target", "Group"])["AUPRC"].mean()
                      .rename("ref").reset_index())
single_means_auroc = (single_long_auroc.groupby(["Target", "Group"])["AUPRC"].mean()
                      .rename("ref").reset_index()
                      if single_long_auroc is not None else None)
plot_loco_panel(loco_long_auroc, "AUPRC", pooled_means_auroc, single_means_auroc,
                nb_loco_auroc, "Predictive performance (AUROC)",
                "loco_cv_panel_auroc", ylim=(0, 1.18), sig_base_y=1.02)

In [ ]:
# Calibration (pooled CV only): calibration-in-the-large (CITL) and slope per
# target, from the stacker's out-of-fold predicted probabilities (notebook 12).
# CITL: intercept of logit(y) ~ offset(logit(p)) — 0 = risk right on average.
# Slope: coefficient of logit(y) ~ logit(p) — 1 = ideal; < 1 = overconfident.
# 95% CIs by subject-level bootstrap. Curves: deciles of predicted risk.
PRED_CSV = PROJECT_ROOT / "outputs" / "predictive_modeling_core" / "pooled" / "per_subject_predictions.csv"
CAL_DIR = PROJECT_ROOT / "outputs" / "predictive_modeling_core" / "calibration"
CAL_DIR.mkdir(parents=True, exist_ok=True)

preds_raw = pd.read_csv(PRED_CSV)
# one prediction per subject: mean probability across the 10 CV repeats
preds = (preds_raw.groupby(["Target", "subject_id"])
         .agg(y_true=("y_true", "first"), p=("meta_proba", "mean"))
         .reset_index())
EPS = 1e-6
preds["p"] = preds["p"].clip(EPS, 1 - EPS)
preds["lp"] = np.log(preds["p"] / (1 - preds["p"]))


def _fit_citl_slope(y, lp):
    """CITL (offset logistic intercept) and calibration slope."""
    from sklearn.linear_model import LogisticRegression
    # slope + intercept model
    lr = LogisticRegression(penalty=None, solver="lbfgs", max_iter=1000)
    lr.fit(lp.reshape(-1, 1), y)
    slope = float(lr.coef_[0, 0])
    # CITL: intercept-only with slope fixed at 1 (offset model, Newton steps)
    a = 0.0
    for _ in range(50):
        eta = a + lp
        mu = 1.0 / (1.0 + np.exp(-eta))
        grad = float(np.sum(y - mu))
        hess = float(np.sum(mu * (1 - mu)))
        step = grad / max(hess, 1e-12)
        a += step
        if abs(step) < 1e-10:
            break
    return a, slope


cal_rows = []
rng = np.random.RandomState(42)
N_BOOT = 1000
for target in [t_ for t_ in VARIABLE_LABELS if t_ in preds["Target"].unique()]:
    sub = preds[preds["Target"] == target]
    y = sub["y_true"].values.astype(int)
    lp = sub["lp"].values
    citl, slope = _fit_citl_slope(y, lp)
    boots = np.empty((N_BOOT, 2))
    for b in range(N_BOOT):
        idx = rng.randint(0, len(y), len(y))
        if len(np.unique(y[idx])) < 2:
            boots[b] = np.nan
            continue
        boots[b] = _fit_citl_slope(y[idx], lp[idx])
    lo, hi = np.nanpercentile(boots, [2.5, 97.5], axis=0)
    cal_rows.append({"Target": target, "N": len(y), "prevalence": float(y.mean()),
                     "mean_pred": float(sub["p"].mean()),
                     "CITL": citl, "CITL_lo": lo[0], "CITL_hi": hi[0],
                     "slope": slope, "slope_lo": lo[1], "slope_hi": hi[1]})
cal_df = pd.DataFrame(cal_rows)
cal_df.to_csv(CAL_DIR / "calibration_stats_pooled.csv", index=False)
print(cal_df.round(3).to_string(index=False))

In [ ]:
# Calibration figure (Fig S3): per-target decile curves + CITL/slope annotation
sns.set_theme(style="whitegrid", font_scale=1.4,
              rc={"font.weight": "bold", "axes.labelweight": "bold",
                  "axes.titleweight": "bold"})
targets_cal = list(cal_df["Target"])
n_cols = 4
n_rows = int(np.ceil(len(targets_cal) / n_cols))
fig, axes = plt.subplots(n_rows, n_cols, figsize=(5.2 * n_cols, 4.8 * n_rows),
                         sharex=True, sharey=True)
axes = np.atleast_1d(axes).ravel()

for ax, target in zip(axes, targets_cal):
    sub = preds[preds["Target"] == target]
    y = sub["y_true"].values.astype(int)
    p = sub["p"].values
    dec = pd.qcut(p, 10, duplicates="drop")
    grp = pd.DataFrame({"y": y, "p": p, "dec": dec}).groupby("dec", observed=True)
    obs = grp["y"].mean().values
    exp = grp["p"].mean().values
    n_g = grp.size().values
    se = np.sqrt(np.clip(obs * (1 - obs), 1e-9, None) / n_g)
    ax.plot([0, 1], [0, 1], color="#888", lw=1.0, ls=":", zorder=1)
    ax.errorbar(exp, obs, yerr=1.96 * se, fmt="o", ms=7, color="#9B7EBD",
                mec="black", mew=0.6, ecolor="#9B7EBD", elinewidth=1.4,
                capsize=3, zorder=3)
    row = cal_df[cal_df["Target"] == target].iloc[0]
    ax.text(0.04, 0.96,
            f"CITL {row.CITL:+.2f} [{row.CITL_lo:+.2f}, {row.CITL_hi:+.2f}]\n"
            f"slope {row.slope:.2f} [{row.slope_lo:.2f}, {row.slope_hi:.2f}]",
            transform=ax.transAxes, ha="left", va="top", fontsize=12,
            fontweight="bold",
            bbox=dict(boxstyle="round,pad=0.3", fc="white", ec="#bbb", lw=0.6))
    ax.set_title(VARIABLE_LABELS[target], fontsize=15, fontweight="bold")
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.set_xlabel("Predicted probability")
    ax.set_ylabel("Observed rate")

for ax in axes[len(targets_cal):]:
    ax.set_visible(False)
plt.tight_layout()
out_png = CAL_DIR / "calibration_curves_pooled.png"
fig.savefig(out_png, dpi=200, bbox_inches="tight")
fig.savefig(out_png.with_suffix(".pdf"), bbox_inches="tight")
fig.savefig(out_png.with_suffix(".svg"), bbox_inches="tight")
print(f"saved {out_png}")

In [ ]:
# LOCO comparison of lesion representations (Fig S15): baseline + infarct
# location / functional LNM / structural LNM per held-out cohort
# (normalised AUPRC and AUROC).

REPS = ["Lesion", "fLNM", "sLNM"]
REP_LABEL = {"Lesion": "infarct location", "fLNM": "functional LNM", "sLNM": "structural LNM"}

loco_scores = pd.read_csv(LOCO / "all_scores.csv")
prev_fold = (loco_scores[loco_scores["Model"] == "Majority"]
             .set_index(["Target", "Fold"])["AUPRC"])
rep = loco_scores[loco_scores["Modality"].isin(REPS)].copy()
_prev = rep.set_index(["Target", "Fold"]).index.map(prev_fold)
rep["normAUPRC"] = (rep["AUPRC"] - _prev) / (1 - _prev)

sns.set_theme(style="whitegrid", font_scale=1.25,
              rc={"font.weight": "bold", "axes.labelweight": "bold"})
targets_present = [t for t in VARIABLE_LABELS if t in rep["Target"].unique()]
off = {"Lesion": -0.26, "fLNM": 0.0, "sLNM": 0.26}
fig, axes = plt.subplots(2, 1, figsize=(18, 11), sharex=True, gridspec_kw={"hspace": 0.12})
for ax, (metric, col, ylab) in zip(axes, [("AUPRC", "normAUPRC", "Normalised AUPRC"),
                                          ("AUROC", "AUROC", "AUROC")]):
    for x, tgt in enumerate(targets_present):
        pv = rep[rep["Target"] == tgt].pivot_table(index="Fold", columns="Modality", values=col).dropna()
        for _, r in pv.iterrows():
            ax.plot([x + off[k] for k in REPS], [r[k] for k in REPS],
                    color="gray", lw=0.6, alpha=0.45, zorder=1)
        for k in REPS:
            ax.scatter(np.full(len(pv), x + off[k]), pv[k], s=34, color=PALETTE[k],
                       edgecolor="white", lw=0.5, zorder=2)
            ax.scatter(x + off[k], pv[k].mean(), marker="D", s=95, color=PALETTE[k],
                       edgecolor="black", lw=1.2, zorder=4)
    for x in range(1, len(targets_present)):
        ax.axvline(x - 0.5, color="gray", lw=0.3, alpha=0.5)
    ax.axhline(0 if metric == "AUPRC" else 0.5, color="black", ls="--", lw=1)
    ax.set_ylabel(ylab, labelpad=8)
    lo, hi = ax.get_ylim()
    ax.set_ylim(lo, hi + 0.06 * (hi - lo))
fig.supylabel("Predictive performance", fontsize=16, fontweight="bold", x=0.05)
axes[1].set_xticks(range(len(targets_present)))
axes[1].set_xticklabels([VARIABLE_LABELS[t] for t in targets_present], fontsize=13)
handles = [Line2D([0], [0], marker="o", color="w", markerfacecolor=PALETTE[k], markersize=10,
                  label=f"Baseline + {REP_LABEL[k]}") for k in REPS] + [
    Line2D([0], [0], marker="D", color="w", markerfacecolor="white", markeredgecolor="black",
           markersize=9, label="Mean across held-out cohorts"),
    Line2D([0], [0], color="gray", lw=1, label="Same held-out cohort")]
axes[1].legend(handles=handles, bbox_to_anchor=(0.5, -0.13), loc="upper center",
               ncol=3, frameon=True, fontsize=12)
out_png = FIG_DIR / "loco_lesion_representations.png"
fig.savefig(out_png, dpi=200, bbox_inches="tight")
fig.savefig(out_png.with_suffix(".pdf"), bbox_inches="tight")
print(f"saved {out_png}")
